# RNN
- X_t : [n sample, d dimmension]
- H_t : [n sample, h hidden state]
- W_xh : [d dimmension, h hidden state]
- W_hh : [h, h]


- X_t * W_xh : [n, h]
- H_t-1 * W_hh : [n, h]

- X_t * W_xh + H_t-1 * W_hh : [n, h]

- num_inputs = số lượng feature của mỗi token
- num_hiddens = kích thước bộ nhớ của model, tức là độ rộng của không gian ẩn, vd = 128 nghĩa là dùng vector 128 chiều để lưu


In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [5]:
class RNN(nn.Module):
    def __init__(self, num_samples, num_hidden, sigma=0.01):
        super().__init__()
        self.num_samples = num_samples
        self.num_hiddens = num_hidden
        self.W_xh = nn.Parameter(torch.rand(num_samples, num_hidden) * sigma)
        self.W_hh = nn.Parameter(torch.rand(num_hidden, num_hidden) * sigma)
        self.b_h = nn.Parameter(torch.zeros(num_hidden))

    def forward(self, inputs, state=None):
        if state is None: # H0, từ đầu tiên chưa có hidden state thì khởi tạo = 0
            # Initial state with shape: (batch_size, num_hiddens)
            state = torch.zeros((inputs.shape[1], self.num_hiddens),
                            device=inputs.device)
        else:
            state, = state
        outputs = []
        for X in inputs:  # Shape of inputs: (num_steps, batch_size, num_inputs)
            state = torch.tanh(torch.matmul(X, self.W_xh) +
                            torch.matmul(state, self.W_hh) + self.b_h)
            outputs.append(state)
        return outputs, state

In [6]:
batch_size, num_inputs, num_hiddens, num_steps = 2, 16, 32, 100
rnn = RNN(num_inputs, num_hiddens)
X = torch.ones((num_steps, batch_size, num_inputs))
outputs, state = rnn(X)

In [ ]:
def check_len(a, n):  #@save
    """Check the length of a list."""
    assert len(a) == n, f'list\'s length {len(a)} != expected length {n}'

def check_shape(a, shape):  #@save
    """Check the shape of a tensor."""
    assert a.shape == shape, \
            f'tensor\'s shape {a.shape} != expected shape {shape}'

check_len(outputs, num_steps)
check_shape(outputs[0], (batch_size, num_hiddens))
check_shape(state, (batch_size, num_hiddens))



# Pytorch

In [3]:
import torch
from torch import nn

class RNN(nn.Module):
    def __init__(self, num_inputs, num_hiddens):
        super().__init__()
        self.num_inputs = num_inputs
        self.num_hiddens = num_hiddens
        self.rnn = nn.RNN(input_size=num_inputs,hidden_size=num_hiddens)

    def forward(self, inputs, H=None):
        return self.rnn(inputs, H)

In [ ]:
class RNNLM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(embed_size, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h):
        emb = self.embedding(x)
        out, h = self.rnn(emb, h)
        out = self.fc(out)
        return out, h

    def init_hidden(self, batch_size):
        return torch.zeros(1, batch_size, hidden_size)

In [2]:
!pip install torchtext

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 35.9 MB/s eta 0:00:00a 0:00:01


In [ ]:
import torch
import torch.nn as nn
from torchtext.datasets import WikiText2
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

tokenizer = get_tokenizer("basic_english")

def yield_tokens(data_iter):
    for text in data_iter:
        yield tokenizer(text)

train_iter = WikiText2(split='train')
vocab = build_vocab_from_iterator(yield_tokens(train_iter), specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

# reload iterator
train_iter = WikiText2(split='train')

In [ ]:
def data_process(raw_text_iter):
    data = []
    for item in raw_text_iter:
        tokens = torch.tensor(vocab(tokenizer(item)), dtype=torch.long)
        data.append(tokens)
    return torch.cat(data)

train_data = data_process(train_iter)

In [ ]:
def batchify(data, batch_size):
    seq_len = data.size(0) // batch_size
    data = data[:seq_len * batch_size]
    data = data.view(batch_size, -1).t().contiguous()
    return data

batch_size = 20
train_data = batchify(train_data, batch_size)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab_size = len(vocab)
embed_size = 100
hidden_size = 128

model = RNNLM(vocab_size, embed_size, hidden_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

seq_len = 35

def get_batch(source, i):
    data = source[i:i+seq_len]
    target = source[i+1:i+1+seq_len]
    return data, target

for epoch in range(3):
    model.train()
    total_loss = 0
    hidden = model.init_hidden(batch_size).to(device)

    for i in range(0, train_data.size(0) - 1, seq_len):
        data, targets = get_batch(train_data, i)

        data = data.to(device)
        targets = targets.to(device).reshape(-1)

        hidden = hidden.detach()

        output, hidden = model(data, hidden)

        output = output.reshape(-1, vocab_size)

        loss = criterion(output, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / (train_data.size(0) // seq_len)
    perplexity = torch.exp(torch.tensor(avg_loss))

    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}, Perplexity: {perplexity:.2f}")